# `collinearity_analysis`

## Purpose

Find pairs of collinear features in the dataset and remove the ones with less predictive power.

## Previous notebook

`single_feature_model_performance`

## Next notebook

`iterative_feature_set_growth`

# Imports

In [1]:
import numpy as np
import pandas as pd

import prepare_data

import os
og_dir = os.getcwd()
os.chdir('../jupyter')
import cutpoint_analysis
os.chdir(og_dir)

import pickle
import time

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.metrics import confusion_matrix, precision_score, recall_score, roc_auc_score, average_precision_score, RocCurveDisplay, accuracy_score

# Load data

In [2]:
train_cohort = prepare_data.load_and_process_cohort('train', 'latest')

/mnt/batch/tasks/shared/LS_root/mounts/clusters/sdrury-compute/code/Users/Stephen_Drury/vps-peds-aki/azure_ml/prepare_data.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cohort[feature] = cohort[feature].fillna(median_impute_val)
/mnt/batch/tasks/shared/LS_root/mounts/clusters/sdrury-compute/code/Users/Stephen_Drury/vps-peds-aki/azure_ml/prepare_data.py:73: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cohort[feature].fillna(cohort[feature+'_for_imputation'], inplace=True)
/mnt/batch/tasks/shared/LS_root/mounts/clusters/sdrury-compute/code/Users/Stephen_Dru

# Load results of single-feature model training

In [3]:
logreg_single_feature_metrics = pd.read_csv('single_feature_logreg_performance_results_latest_lab_imp.csv')
svc_single_feature_metrics = pd.read_csv('single_feature_svc_performance_results_latest_lab_imp.csv')

In [4]:
logreg_single_feature_metrics['auroc_logreg'] = logreg_single_feature_metrics['auroc'].copy()
logreg_single_feature_metrics['auprc_logreg'] = logreg_single_feature_metrics['auprc'].copy()
logreg_single_feature_metrics.drop(['auroc', 'auprc'], axis=1, inplace=True)

svc_single_feature_metrics['auroc_svc'] = svc_single_feature_metrics['auroc'].copy()
svc_single_feature_metrics['auprc_svc'] = svc_single_feature_metrics['auprc'].copy()
svc_single_feature_metrics.drop(['auroc', 'auprc', 'model_type'], axis=1, inplace=True)

combined_single_feature_metrics = svc_single_feature_metrics.merge(
    logreg_single_feature_metrics,
    on='input_feature', 
    how='outer'
)

In [5]:
combined_single_feature_metrics.sort_values('auroc_logreg', ascending=False).head()

,input_feature,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
77,CREATININE_median,0.553595,0.051064,0.749084,0.155936
76,CREATININE_mean,0.447742,0.024243,0.748747,0.156311
75,CREATININE_max,0.563009,0.049463,0.747516,0.155736
78,CREATININE_min,0.467392,0.025293,0.745064,0.154068
212,RDW_min,0.335839,0.022809,0.683738,0.059513


In [6]:
combined_single_feature_metrics.sort_values('auprc_logreg', ascending=False).head()

,input_feature,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
219,sbp_median,0.333553,0.020938,0.667237,0.189504
218,sbp_mean,0.326342,0.019412,0.664335,0.184398
151,mbp_median,0.643073,0.045762,0.661855,0.171429
150,mbp_mean,0.655862,0.044336,0.659997,0.171156
146,map_mean,0.686057,0.056105,0.663901,0.165969


In [7]:
combined_single_feature_metrics.sort_values('auroc_svc', ascending=False).head()

,input_feature,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
146,map_mean,0.686057,0.056105,0.663901,0.165969
152,mbp_min,0.674779,0.127004,0.669871,0.134087
153,MCV_max,0.669991,0.091450,0.678203,0.142019
209,RDW_max,0.666473,0.056102,0.654358,0.043023
150,mbp_mean,0.655862,0.044336,0.659997,0.171156


In [8]:
combined_single_feature_metrics.sort_values('auprc_svc', ascending=False).head()

,input_feature,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
152,mbp_min,0.674779,0.127004,0.669871,0.134087
149,mbp_max,0.651677,0.109848,0.641148,0.139172
153,MCV_max,0.669991,0.091450,0.678203,0.142019
115,HGB_max,0.603867,0.072459,0.610673,0.071466
146,map_mean,0.686057,0.056105,0.663901,0.165969


## Check to make sure there are no missing values in the metrics for single-feature models

In [10]:
for col in combined_single_feature_metrics.columns:
    print('~'*15)
    print(col)
    print(combined_single_feature_metrics[col].isna().sum())

~~~~~~~~~~~~~~~
input_feature
0
~~~~~~~~~~~~~~~
auroc_svc
0
~~~~~~~~~~~~~~~
auprc_svc
0
~~~~~~~~~~~~~~~
auroc_logreg
0
~~~~~~~~~~~~~~~
auprc_logreg
0


## Remove non-feature columns from dataframes

In [11]:
feature_cols = prepare_data.get_feature_cols(train_cohort)

train_cohort = train_cohort[feature_cols + ['aki_72hrs_any']]
# val_cohort = val_cohort[feature_cols + ['aki_72hrs_any']]

# Find collinearity among feature pairs

## Obtain correlation matrices

In [12]:
feature_corr_dict = dict()

feature_corr_dict['pearson'] = train_cohort[feature_cols].corr(method='pearson')
feature_corr_dict['kendall'] = train_cohort[feature_cols].corr(method='kendall')
feature_corr_dict['spearman'] = train_cohort[feature_cols].corr(method='spearman')

/anaconda/envs/azureml_py38/lib/python3.8/site-packages/scipy/stats/_stats_py.py:5278: RuntimeWarning: overflow encountered in long_scalars
  (2 * xtie * ytie) / m + x0 * y0 / (9 * m * (size - 2)))


## Reformat so that each row is one feature pair

In [13]:
def reformat_correlation_matrix(corr_mtx):
    corr_mtx = corr_mtx.reindex(sorted(corr_mtx.columns), axis=1)
    corr_mtx = corr_mtx.sort_index()
    return corr_mtx

def get_pairwise_correlation_dataframe(corr_mtx, corr_col_name='corr'):
    features_used = []
    pairwise_corr_dict_list = []
    
    corr_mtx = reformat_correlation_matrix(corr_mtx)
    
    for feature1 in corr_mtx.columns:
        for feature2 in corr_mtx.columns:
            if feature2 not in features_used and feature2 != feature1:
                corr_value = corr_mtx[feature1].loc[feature2]
                temp_dict = {
                    'feature1': feature1,
                    'feature2': feature2,
                    corr_col_name: corr_value
                }
                pairwise_corr_dict_list.append(temp_dict)
    
        features_used.append(feature1)
    
    pairwise_corr_df = pd.DataFrame(pairwise_corr_dict_list)

    return pairwise_corr_df

In [14]:
pairwise_corr_dict = dict()

for key in feature_corr_dict.keys():
    pairwise_corr_dict[key] = get_pairwise_correlation_dataframe(feature_corr_dict[key])

In [15]:
pairwise_corr_dict['pearson']

,feature1,feature2,corr
0,ALBUMIN_max,ALBUMIN_mean,1.000000
1,ALBUMIN_max,ALBUMIN_median,1.000000
2,ALBUMIN_max,ALBUMIN_min,0.999999
3,ALBUMIN_max,ALC_max,0.003574
4,ALBUMIN_max,ALC_mean,0.003406
...,...,...,...
27961,weight_max,weight_median,0.955442
27962,weight_max,weight_min,0.781208
27963,weight_mean,weight_median,0.999614
27964,weight_mean,weight_min,0.928936


In [16]:
pairwise_corr_dict.keys()

dict_keys(['pearson', 'kendall', 'spearman'])

## Find all feature pairs with collinearity greater than threshold value of 0.8

In [17]:
def get_collinear_pairs_df(corr_mtx_dict, corr_thresh=0.8):
    # pairwise_corr_mtx_dict = dict()
    pairwise_corr_df = pd.DataFrame()
    for key in corr_mtx_dict.keys():
        pairwise_corr_mtx = corr_mtx_dict[key] # get_pairwise_correlation_dataframe(corr_mtx_dict[key], key)
        pairwise_corr_mtx['corr_' + key + '_abs'] = [abs(x) for x in pairwise_corr_mtx['corr']]
        if len(pairwise_corr_df) == 0:
            pairwise_corr_df = pairwise_corr_mtx.drop('corr', axis=1).copy()
        else:
            pairwise_corr_df = pairwise_corr_df.merge(
                pairwise_corr_mtx.drop('corr', axis=1),
                on=['feature1', 'feature2'],
            )
    pairwise_corr_df['max_abs_corr'] = pairwise_corr_df[['corr_' + key + '_abs' for key in corr_mtx_dict.keys()]].max(axis=1)
    collinear_pairs_df = pairwise_corr_df[pairwise_corr_df['max_abs_corr'] > corr_thresh]

    return collinear_pairs_df

In [18]:
pairwise_corr_df = get_collinear_pairs_df(pairwise_corr_dict, corr_thresh=0.8)

In [19]:
pairwise_corr_df.head()

,feature1,feature2,corr_pearson_abs,corr_kendall_abs,corr_spearman_abs,max_abs_corr
0,ALBUMIN_max,ALBUMIN_mean,1.000000,0.986988,0.993006,1.000000
1,ALBUMIN_max,ALBUMIN_median,1.000000,0.986812,0.992783,1.000000
2,ALBUMIN_max,ALBUMIN_min,0.999999,0.960538,0.974906,0.999999
236,ALBUMIN_mean,ALBUMIN_median,1.000000,0.999351,0.999745,1.000000
237,ALBUMIN_mean,ALBUMIN_min,1.000000,0.973972,0.983869,1.000000


## For each collinear feature pair, compare results of single-feature models and keep the feature that performs better (if all metrics agree on better feature)

In [20]:
def get_better_feature_from_pair(feature1, feature2, results_df, model_key, metric_key):
    feature1_metric_val = results_df[results_df['input_feature']==feature1].iloc[0][metric_key + '_' + model_key]
    feature2_metric_val = results_df[results_df['input_feature']==feature2].iloc[0][metric_key + '_' + model_key]
    if feature1_metric_val > feature2_metric_val:
        return 1
    else:
        return 2

In [21]:
model_keys = ['logreg', 'svc']
metric_keys = ['auroc', 'auprc']

better_feature_list_dict = dict()

for model_key in model_keys:
    better_feature_list_dict[model_key] = []
    for metric_key in metric_keys:
        for i in range(len(pairwise_corr_df)):
            feature1 = pairwise_corr_df.iloc[i]['feature1']
            feature2 = pairwise_corr_df.iloc[i]['feature2']
            try:
                temp_dict = {
                    'model': model_key,
                    'metric': metric_key,
                    'feature1': feature1,
                    'feature2': feature2,
                    'better_feature': get_better_feature_from_pair(feature1, feature2, combined_single_feature_metrics, model_key, metric_key)
                }
                better_feature_list_dict[model_key].append(temp_dict)
            except:
                print('~'*15)
                print(feature1)
                print(feature2)

better_feature_df_logreg = pd.DataFrame(better_feature_list_dict['logreg'])
better_feature_df_svc = pd.DataFrame(better_feature_list_dict['svc'])

In [22]:
better_feature_df_logreg.head()

,model,metric,feature1,feature2,better_feature
0,logreg,auroc,ALBUMIN_max,ALBUMIN_mean,1
1,logreg,auroc,ALBUMIN_max,ALBUMIN_median,1
2,logreg,auroc,ALBUMIN_max,ALBUMIN_min,1
3,logreg,auroc,ALBUMIN_mean,ALBUMIN_median,2
4,logreg,auroc,ALBUMIN_mean,ALBUMIN_min,2


In [23]:
better_feature_df_logreg_pivot = better_feature_df_logreg.pivot(
    index=['feature1', 'feature2'], 
    columns=['model', 'metric'], 
    values='better_feature'
)

better_feature_df_logreg_pivot.columns = ['better_feature_' + '_'.join(tup) for tup in better_feature_df_logreg_pivot.columns.to_flat_index()]

better_feature_df_logreg_pivot.reset_index(inplace=True)

better_feature_df_svc_pivot = better_feature_df_svc.pivot(
    index=['feature1', 'feature2'], 
    columns=['model', 'metric'], 
    values='better_feature'
)

better_feature_df_svc_pivot.columns = ['better_feature_' + '_'.join(tup) for tup in better_feature_df_svc_pivot.columns.to_flat_index()]

better_feature_df_svc_pivot.reset_index(inplace=True)

better_feature_df = better_feature_df_svc_pivot.merge(
    better_feature_df_logreg_pivot,
    on=['feature1', 'feature2'],
    how='outer'
)

better_feature_df

,feature1,feature2,better_feature_svc_auroc,better_feature_svc_auprc,better_feature_logreg_auroc,better_feature_logreg_auprc
0,ALBUMIN_max,ALBUMIN_mean,2,1,1,2
1,ALBUMIN_max,ALBUMIN_median,1,2,1,2
2,ALBUMIN_max,ALBUMIN_min,2,1,1,2
3,ALBUMIN_mean,ALBUMIN_median,1,2,2,1
4,ALBUMIN_mean,ALBUMIN_min,1,1,2,2
...,...,...,...,...,...,...
360,weight_max,weight_median,2,2,2,1
361,weight_max,weight_min,1,1,1,1
362,weight_mean,weight_median,1,2,2,2
363,weight_mean,weight_min,1,1,1,2


In [24]:
better_feature_df = better_feature_df.merge(
    pairwise_corr_df[['feature1', 'feature2', 'max_abs_corr']],
    on=['feature1', 'feature2'],
    how='left'
).sort_values('max_abs_corr', ascending=False)

In [25]:
better_feature_df

,feature1,feature2,better_feature_svc_auroc,better_feature_svc_auprc,better_feature_logreg_auroc,better_feature_logreg_auprc,max_abs_corr
195,LIPASE_mean,LIPASE_median,2,2,2,2,1.000000
127,ESR_mean,ESR_median,2,2,2,2,1.000000
133,FERRITIN_mean,FERRITIN_median,2,2,2,2,1.000000
121,D_DIMER_mean,D_DIMER_median,1,1,1,1,1.000000
274,PROCALCITONIN_mean,PROCALCITONIN_median,1,1,2,2,1.000000
...,...,...,...,...,...,...,...
87,CALCIUM_ION_median,CALCIUM_ION_min,1,1,2,2,0.805875
84,CALCIUM_ION_max,CALCIUM_ION_median,1,1,1,1,0.802667
313,dbp_mean,map_median,1,2,2,2,0.802303
316,dbp_mean,mbp_min,2,2,2,1,0.800509


In [26]:
better_feature_df['svc_agrees'] = (better_feature_df['better_feature_svc_auroc'] == better_feature_df['better_feature_svc_auprc']).astype(int)
better_feature_df['logreg_agrees'] = (better_feature_df['better_feature_logreg_auroc'] == better_feature_df['better_feature_logreg_auprc']).astype(int)
better_feature_df['auroc_agrees'] = (better_feature_df['better_feature_svc_auroc'] == better_feature_df['better_feature_logreg_auroc']).astype(int)
better_feature_df['auprc_agrees'] = (better_feature_df['better_feature_svc_auprc'] == better_feature_df['better_feature_logreg_auprc']).astype(int)

better_feature_df['all_agree'] = better_feature_df['svc_agrees'] * better_feature_df['logreg_agrees'] * better_feature_df['auroc_agrees'] * better_feature_df['auprc_agrees']

In [27]:
len(better_feature_df[better_feature_df['all_agree']==1]) / len(better_feature_df)

0.336986301369863

In [28]:
better_feature_all_agree_df = better_feature_df[better_feature_df['all_agree']==1].copy()

better_feature_all_agree_df['worse_feature_col'] = ['feature%d' % (1 if n==2 else 2) for n in better_feature_all_agree_df['better_feature_svc_auroc']]
better_feature_all_agree_df['worse_feature'] = [
    better_feature_all_agree_df.iloc[i][better_feature_all_agree_df.iloc[i]['worse_feature_col']] for i in range(len(better_feature_all_agree_df))
]

In [29]:
better_feature_all_agree_df.head()

,feature1,feature2,better_feature_svc_auroc,better_feature_svc_auprc,better_feature_logreg_auroc,better_feature_logreg_auprc,max_abs_corr,svc_agrees,logreg_agrees,auroc_agrees,auprc_agrees,all_agree,worse_feature_col,worse_feature
195,LIPASE_mean,LIPASE_median,2,2,2,2,1.0,1,1,1,1,1,feature1,LIPASE_mean
127,ESR_mean,ESR_median,2,2,2,2,1.0,1,1,1,1,1,feature1,ESR_mean
133,FERRITIN_mean,FERRITIN_median,2,2,2,2,1.0,1,1,1,1,1,feature1,FERRITIN_mean
121,D_DIMER_mean,D_DIMER_median,1,1,1,1,1.0,1,1,1,1,1,feature2,D_DIMER_median
124,ESR_max,ESR_mean,2,2,2,2,1.0,1,1,1,1,1,feature1,ESR_max


In [30]:
better_feature_all_agree_df[
    (better_feature_all_agree_df['feature1']=='map_median') | (better_feature_all_agree_df['feature2']=='map_median')
]

,feature1,feature2,better_feature_svc_auroc,better_feature_svc_auprc,better_feature_logreg_auroc,better_feature_logreg_auprc,max_abs_corr,svc_agrees,logreg_agrees,auroc_agrees,auprc_agrees,all_agree,worse_feature_col,worse_feature
317,dbp_median,map_median,2,2,2,2,0.800363,1,1,1,1,1,feature1,dbp_median


In [31]:
features_to_remove_list = []

for i in range(len(better_feature_all_agree_df)):
    feature1 = better_feature_all_agree_df.iloc[i]['feature1']
    feature2 = better_feature_all_agree_df.iloc[i]['feature2']
    worse_feature = better_feature_all_agree_df.iloc[i]['worse_feature']
    if feature1 not in features_to_remove_list and feature2 not in features_to_remove_list:
        features_to_remove_list.append(worse_feature)

In [32]:
len(features_to_remove_list)

80

In [33]:
combined_single_feature_metrics.describe()

,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
count,237.000000,237.000000,237.000000,237.000000
mean,0.502937,0.032943,0.549007,0.051105
std,0.067891,0.011806,0.063445,0.036139
min,0.317878,0.019412,0.407320,0.027571
25%,0.466443,0.027051,0.507798,0.033063
50%,0.497681,0.029370,0.536287,0.037421
75%,0.548555,0.036199,0.572503,0.048594
max,0.686057,0.127004,0.749084,0.189504


In [34]:
features_to_remove_metrics_df = combined_single_feature_metrics[combined_single_feature_metrics['input_feature'].isin(features_to_remove_list)].copy()
features_to_remove_metrics_df

,input_feature,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
6,ALC_mean,0.474491,0.026470,0.493766,0.035748
11,ALP_median,0.460662,0.026155,0.469323,0.030656
17,AMYLASE_max,0.487629,0.027636,0.497168,0.030923
21,ANC_max,0.399444,0.022786,0.407320,0.028079
22,ANC_mean,0.422818,0.024010,0.407436,0.028179
...,...,...,...,...,...
223,SODIUM_median,0.541754,0.038632,0.537076,0.040621
225,temp_max,0.490955,0.028545,0.506907,0.029354
230,WBC_mean,0.464948,0.032049,0.464942,0.030977
231,WBC_median,0.540366,0.047900,0.463874,0.030923


In [35]:
features_to_remove_metrics_df.sort_values('auprc_logreg', ascending=False).head(50)

,input_feature,auroc_svc,auprc_svc,auroc_logreg,auprc_logreg
218,sbp_mean,0.326342,0.019412,0.664335,0.184398
78,CREATININE_min,0.467392,0.025293,0.745064,0.154068
84,dbp_mean,0.428044,0.022739,0.652394,0.148770
85,dbp_median,0.317878,0.019858,0.650096,0.142758
145,map_max,0.407803,0.023550,0.637434,0.135516
86,dbp_min,0.348867,0.022596,0.645992,0.105753
113,GLUCOSE_median,0.550120,0.031119,0.578729,0.065608
116,HGB_mean,0.559205,0.033173,0.571996,0.056643
117,HGB_median,0.567378,0.055187,0.568976,0.056369
171,PH_median,0.397058,0.022909,0.602068,0.054518


In [36]:
with open('pickle/removed_collinear_features.pickle', 'wb') as outfile:
    pickle.dump(list(features_to_remove_metrics_df['input_feature']), outfile)

In [37]:
len(list(features_to_remove_metrics_df['input_feature']))

80